In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.16 Time-Dependent Density-Functional Theory

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.16",
    title="Time-Dependent Density-Functional Theory",
    blurb="The exact two-electron laboratory of §8.2 is set in motion. A "
    "momentum kick plus Crank–Nicolson propagation turns one dipole "
    "signal into a complete optical spectrum whose peaks land on the "
    "exact excitation energies to 2×10⁻⁴ — with the triplet and "
    "parity-forbidden states correctly, measurably silent. The same "
    "kick is then run through the two workhorse approximations: "
    "time-dependent Hartree–Fock blue-shifts the first excitation by "
    "3% (missing correlation, quantified), while orbital-free Hartree — "
    "self-interaction left in — wrecks the spectrum beyond recognition. "
    "Approximate quantum dynamics, judged against exact quantum "
    "dynamics, on the one system where both are affordable.",
    difficulty="advanced",
    estimate="150–180 min",
)

## Notebook overview

[§8.15](optics-excitons.ipynb) computed spectra from *eigenstates* —
linear response assembled state by state. Real electron dynamics is
richer: attosecond pulses, strong fields, charge transfer — regimes
where one wants to *propagate* electrons in time. The exact
time-dependent Schrödinger equation for $N$ electrons is as intractable
as the static one, and the resolution is the same maneuver that built
[§8.6](hohenberg-kohn.ipynb): Runge and Gross {cite}`rungegross1984`
proved that the time-dependent *density* $n(x, t)$ determines the
time-dependent potential, licensing a time-dependent Kohn–Sham scheme.
But where static DFT has [§8.8](exact-conditions-band-gap.ipynb)'s
well-mapped failure modes, time-dependent functionals are younger and
rougher terrain {cite}`ullrich2012` — which makes an *exact testing
ground* worth everything. This volume owns one.

The build: [§8.2](exact-laboratory.ipynb)'s two-electron soft-Coulomb
atom returns, its full $(x_1, x_2)$ wavefunction now *propagated* by
Crank–Nicolson (the unconditionally unitary scheme of
[§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb),
upgraded to a sparse LU factorization reused across four thousand
steps — norm conserved to $10^{-8}$ over the whole run). The
spectroscopy protocol is the standard one of real-time TDDFT codes: a
weak momentum kick $e^{\mathrm i\kappa X}$ at $t = 0$, the dipole
$\langle X\rangle(t)$ recorded, its Fourier transform read as the
absorption spectrum. Certification first: the exact propagation's peaks
must land on the exact eigenstate differences — they do, at
$2\times10^{-4}$ — and the states that symmetry forbids must be
*absent*: the triplet at $\Delta E = 0.422$ (spin symmetry preserved by
propagation) and the even-parity singlet at $0.615$ (dipole parity),
both measurably silent. Linearity is verified by halving the kick
(response ratio $2.00$). Then the judgment: the same kick through
time-dependent Hartree–Fock (for two electrons, exact exchange —
adiabatic, correlation-free) blue-shifts the first peak to $0.550$
against the exact $0.534$, a $3\%$ error the notebook prices; and
through time-dependent *Hartree* — self-interaction uncorrected — which
scatters spurious structure across the spectrum, the dynamical face of
the self-interaction disease [§8.8](exact-conditions-band-gap.ipynb)
diagnosed statically. One system, three dynamics, every claim gated.

> **Conventions (this notebook).** Atomic units. The laboratory is
> [§8.2](exact-laboratory.ipynb)'s: $v(x) = -2/\sqrt{x^2+1}$,
> interaction $w(u) = 1/\sqrt{u^2+1}$, grid $x \in [-10, 10]$ with
> $N = 121$ points (spacing $1/6$), two electrons in a spin singlet
> (symmetric spatial wavefunction). Sparse Hamiltonians as in
> [§8.2](exact-laboratory.ipynb)
> (Kronecker sums, `scipy.sparse`); statics by `eigsh`; dynamics by
> Crank–Nicolson with `scipy.sparse.linalg.splu` factorized once
> ($\Delta t = 0.05$, $4000$ steps). Spectra: kick $\kappa = 10^{-3}$,
> Hann window, zero-padded `numpy.fft.rfft` of
> $\langle X\rangle(t) - \langle X\rangle(0)$.
>
> **How to read the checks.** Each exercise closes with a `validate`
> call against an independent fact: a conservation law, an eigenstate
> difference, a selection rule, a linearity ratio. A ✓ is strong
> evidence; a ✗ is a prompt to locate the discrepancy, not an
> automatic verdict.
>
> **Scope.** Real-time propagation and kick spectroscopy for one
> closed-shell pair, with time-dependent Hartree and Hartree–Fock as
> the approximate dynamics (for two singlet electrons TDHF *is*
> time-dependent exact exchange, so the comparison isolates adiabatic
> correlation). Production TDDFT — ALDA and beyond, memory effects,
> double excitations, Casida's matrix formulation — is Ullrich's
> monograph {cite}`ullrich2012`.

## Theory in brief

### Runge–Gross, and why propagation is spectroscopy

The Runge–Gross theorem {cite}`rungegross1984` is the time-dependent
twin of [§8.6](hohenberg-kohn.ipynb): two potentials differing by more
than $c(t)$ cannot produce the same evolving density from the same
initial state — so $n(x, t)$ is, in principle, a complete record of the
dynamics. The practical protocol reads the record spectroscopically.
Kick the ground state with a uniform momentum boost,

```{math}
:label: eq-td-kick
\Psi(t{=}0^+) = e^{\mathrm i\kappa X}\,\Psi_0,
\qquad X = x_1 + x_2 ,
```

which populates every dipole-connected eigenstate with amplitude
$\propto \kappa\,\langle n|X|0\rangle$. The dipole then rings,

```{math}
:label: eq-td-dipole
\langle X\rangle(t) - \langle X\rangle(0) \;=\;
2\kappa \sum_n \big|\langle n|X|0\rangle\big|^2
\sin\!\big(\omega_{n0}t\big) + O(\kappa^2) ,
```

so the Fourier transform of one real-time signal *is* the absorption
spectrum: peak positions at exact excitation energies, weights at
dipole strengths — precisely how real-time TDDFT codes compute optical
spectra of molecules. The kick must be weak (Eq. {eq}`eq-td-dipole` is
first order in $\kappa$), and weakness is checkable: halve $\kappa$,
the response must halve.

### Crank–Nicolson, unitary by construction

Propagation uses the scheme of
[§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb):

```{math}
:label: eq-td-cn
\Big(\mathbb 1 + \tfrac{\mathrm i\Delta t}{2} H\Big)\,\Psi_{t+\Delta t}
\;=\; \Big(\mathbb 1 - \tfrac{\mathrm i\Delta t}{2} H\Big)\,\Psi_t ,
```

exactly unitary for Hermitian $H$ (the two factors are conjugate), so
the norm is conserved to solver precision no matter how long the run.
For the exact problem $H$ is time-independent and the sparse LU
factorization of the left-hand matrix is computed *once*; for the
self-consistent dynamics (TDH/TDHF) the potential depends on the
evolving density and the linear system is rebuilt each step.

### The two approximate dynamics

Both electrons share one orbital $\varphi(x, t)$; the approximations
differ in the mean field it feels:

```{math}
:label: eq-td-meanfield
\mathrm i\,\partial_t\varphi = \Big[-\tfrac12\partial_x^2 + v(x)
+ c\,v_{\mathrm H}[n](x, t)\Big]\varphi,
\qquad n = 2|\varphi|^2 ,
```

with $c = \tfrac12$ for **TDHF** (the exchange integral for a doubly
occupied orbital cancels exactly half the Hartree term — each electron
feels only the *other* one; self-interaction-free, correlation-free)
and $c = 1$ for **TD-Hartree** (each electron repels the full density,
*itself included* — the self-interaction error of
[§8.8](exact-conditions-band-gap.ipynb), now propagating). Adiabatic
TDDFT with an exact-exchange functional coincides with TDHF for this
system, so the TDHF-versus-exact gap below *is* the adiabatic
exchange-only TDDFT error.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.sparse import diags, identity, kron
from scipy.sparse.linalg import eigsh, splu

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

L_BOX, N_GRID = 10.0, 121
x_grid = np.linspace(-L_BOX, L_BOX, N_GRID)
DX = x_grid[1] - x_grid[0]

lap = diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(N_GRID, N_GRID)) / DX**2
KIN_1D = -0.5 * lap
V_EXT = -2.0 / np.sqrt(x_grid**2 + 1.0)
H_1D = KIN_1D + diags(V_EXT)

X1_MESH, X2_MESH = np.meshgrid(x_grid, x_grid, indexing="ij")
W_EE = 1.0 / np.sqrt((X1_MESH - X2_MESH) ** 2 + 1.0)
EYE = identity(N_GRID)
H_2E = (kron(H_1D, EYE) + kron(EYE, H_1D) + diags(W_EE.ravel())).tocsc()
X_TOTAL = (X1_MESH + X2_MESH).ravel()

# soft-Coulomb kernel matrix for the mean-field dynamics
W_KERNEL = 1.0 / np.sqrt((x_grid[:, None] - x_grid[None, :]) ** 2 + 1.0)


def dipole_spectrum(signal, dt, w_max=1.2):
    """Kick-spectroscopy spectrum: |FFT| of the windowed dipole signal.

    Hann window against leakage, 8x zero padding for peak resolution.

    Parameters
    ----------
    signal : numpy.ndarray
        Dipole record <X>(t).
    dt : float
        Time step.
    w_max : float, optional
        Upper frequency cut for the returned arrays.

    Returns
    -------
    tuple
        (frequency grid, spectral magnitude).
    """
    centered = signal - signal[0]
    padded = 8 * len(centered)
    amp = np.abs(np.fft.rfft(centered * np.hanning(len(centered)), padded))
    freq = 2.0 * np.pi * np.fft.rfftfreq(padded, dt)
    sel = freq < w_max
    return freq[sel], amp[sel]

## Exercise 1 — The laboratory, reopened and classified

Before dynamics, statics: the exact spectrum, with every state's
symmetry papers in order.

**Part a)** Diagonalize the two-electron Hamiltonian (`eigsh`, eight
lowest states) and classify each by *exchange* symmetry
($\psi(x_1, x_2) = \pm\psi(x_2, x_1)$: singlet/triplet spatial parts,
[§6.20](../06-quantum-mechanics/identical-particles.ipynb)) and by
*parity* ($\psi(-x_1, -x_2) = \pm\psi$). Ground state $E_0 = -2.2391$,
symmetric and even, as [§8.2](exact-laboratory.ipynb) built it.

**Part b)** Mark the dipole-active states. The kick operator $X$ is
symmetric under exchange and odd under parity, so from the ground
state only *singlet, odd-parity* states can light up: states $2$
($\Delta E = 0.5339$) and $6$ ($\Delta E = 0.7147$) among the eight.
The triplet at $0.4220$ and the even singlet at $0.6145$ are on the
forbidden list — Exercise 3 checks that the dynamics respects it.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — papers in order

The ground energy, the two allowed excitations, and a spectrum that is
neither all-singlet nor all-triplet (the classifier actually
classifies).

In [ ]:
validate.close(e_ground, -2.2391, "ground energy of the exact laboratory", atol=1e-3)
validate.close(active_energies[0], 0.5339, "first dipole-active excitation", atol=1e-3)
validate.close(active_energies[1], 0.7147, "second dipole-active excitation", atol=1e-3)
validate.check(
    bool(2 <= n_singlets <= 6 and len(active_energies) == 2),
    "classifier separates singlets, triplets, parities",
    f"{n_singlets} singlets, 2 dipole-active",
)

## Exercise 2 — Exact dynamics: kicked, unitary, linear

The full $(x_1, x_2)$ wavefunction in real time.

**Part a)** Kick with $\kappa = 10^{-3}$ (Eq. {eq}`eq-td-kick`) and
propagate $4000$ Crank–Nicolson steps of $\Delta t = 0.05$ (Eq.
{eq}`eq-td-cn`), factorizing the sparse left-hand matrix *once* with
`scipy.sparse.linalg.splu` — the whole $200$-a.u. run in seconds.
Record $\langle X\rangle(t)$.

**Part b)** Certify unitarity: the norm after $4000$ steps must equal
$1$ to $10^{-8}$ — Crank–Nicolson's defining virtue, not a numerical
accident.

**Part c)** Certify linearity: rerun with $\kappa/2$ and confirm the
dipole response amplitude halves (ratio $2.00$ within $1\%$) — the
spectroscopy below is measuring the *linear* response, as Eq.
{eq}`eq-td-dipole` requires.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — unitary and linear, certified

Norm conservation at $10^{-8}$ over $4000$ steps; response linear in
the kick at $1\%$.

In [ ]:
validate.check(
    bool(abs(norm_final - 1.0) < 1e-8),
    "Crank-Nicolson conserves the norm over the whole run",
    f"drift {abs(norm_final-1.0):.1e}",
)
validate.close(lin_ratio, 2.0, "response linear in the kick", rtol=1e-2)

## Exercise 3 — Spectroscopy: peaks where allowed, silence where not

Fourier-transform the dipole and hold it against Exercise 1.

**Part a)** The two dominant peaks must land on the two dipole-active
eigenstate differences at $2\times10^{-4}$ — real-time propagation and
static diagonalization are two routes to one spectrum, and here both
are exact.

**Part b)** The *forbidden* lines must be silent: spectral weight
within $\pm0.02$ of the triplet ($0.4220$) and of the even-parity
singlet ($0.6145$) at least fivefold below even the *weaker* allowed
line (which carries $19\%$ of the main peak) — spin symmetry and
parity, conserved by the propagator, acting as selection rules in the
dynamics itself. (The $2.5\%$ residual at the even window is Hann
leakage from the strong $0.534$ line at this run length, not a
transition: the signature is that it has no peak of its own.)

In [ ]:
# (solution hidden on the public site)


### Validation 3 — two routes, one spectrum

Peaks on the eigenstate differences; forbidden windows far below the
weaker allowed line.

In [ ]:
validate.check(
    bool(peak_dev < 2e-3),
    "propagation peaks = static excitation energies",
    f"worst deviation {peak_dev:.1e}",
)
validate.check(
    bool(
        allowed_second > 0.15
        and silent_triplet < allowed_second / 5.0
        and silent_even < allowed_second / 5.0
    ),
    "forbidden lines fivefold below the weaker allowed line",
    f"{100*silent_triplet:.2f}%, {100*silent_even:.2f}% vs allowed {100*allowed_second:.1f}%",
)

## Exercise 4 — TDHF: adiabatic exact exchange, priced

Now the approximate dynamics, starting with the best mean field on
offer.

**Part a)** Converge the restricted HF ground state (one doubly
occupied orbital; $c = \tfrac12$ in Eq. {eq}`eq-td-meanfield`,
mixing-damped SCF as in [§8.3](hartree-fock-atoms.ipynb)):
$E^{\mathrm{HF}} = -2.2250$, leaving $14.1$ mHa of correlation
against the exact $-2.2391$ — [§8.2](exact-laboratory.ipynb)'s
percent-level verdict on HF, reproduced in one dimension.

**Part b)** Kick the HF orbital and propagate with the
*self-consistent* Crank–Nicolson (the mean field is rebuilt from
$|\varphi(t)|^2$ every step — the linear system too). The first
spectral peak lands at $0.550$: blue-shifted from the exact $0.534$ by
$3.0\%$. That shift *is* the adiabatic exchange-only TDDFT error for
this system — correlation missing from both the ground state and the
response — and now it has a number.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — correlation's price tag

The HF ground energy and its $14$ mHa deficit; the TDHF peak
blue-shifted by $3\%$ — small, systematic, and now measured.

In [ ]:
validate.close(e_hf, -2.2250, "RHF ground energy", atol=1e-3)
validate.check(
    bool(-18.0 < corr_mha < -10.0),
    "correlation energy ~ -14 mHa (percent level, as in 8.2)",
    f"{corr_mha:.1f} mHa",
)
validate.check(
    bool(0.545 < hf_peak < 0.555 and 0.015 < hf_shift < 0.045),
    "TDHF blue-shifts the excitation by ~3%",
    f"peak {hf_peak:.4f}, shift {100*hf_shift:+.1f}%",
)

## Exercise 5 — Verdict: the self-interaction catastrophe, and the scoreboard

The same dynamics with $c = 1$: each electron now repels the *full*
density, itself included.

**Part a)** Converge and kick the TD-Hartree system. Its spectrum is
not shifted — it is *wrecked*: the strongest line lands near $0.20$
— $62\%$ below the exact excitation — with further spurious structure
at $0.39$ and $0.63$, none of it corresponding to any exact
transition. The
self-interaction error that [§8.8](exact-conditions-band-gap.ipynb)
showed bending static curves here detunes the dynamics entirely: the
electron drags a phantom copy of itself through every oscillation.

**Part b)** Assemble the verdict figure — exact, TDHF, TD-Hartree
spectra against the exact excitation lines — and the scoreboard table:
method, ground energy, first-peak position, error. Two lessons, both
measured: *the mean field's quality decides the dynamics' quality*
(TDHF within $3\%$, Hartree off by tens of percent), and *removing
self-interaction is not optional* — the same two rules that govern
every functional on [§8.8](exact-conditions-band-gap.ipynb)'s static
battleground.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — wrecked, measurably

TD-Hartree far off where TDHF is close; the ordering of errors is the
lesson, and it is gated.

In [ ]:
validate.check(
    bool(abs(hartree_err) > 0.15),
    "self-interaction detunes TD-Hartree by tens of percent",
    f"error {100*hartree_err:+.1f}%",
)
validate.check(
    bool(abs(hf_shift) < 0.05 < abs(hartree_err)),
    "mean-field quality decides dynamical quality (TDHF << Hartree error)",
    f"{100*abs(hf_shift):.1f}% vs {100*abs(hartree_err):.1f}%",
)
validate.check(
    bool(e_hartree > e_hf),
    "self-interaction raises the mean-field ground energy",
    f"{e_hartree:.4f} > {e_hf:.4f}",
)

:::{admonition} With your assistant
:class: tip
Everything here was *linear* response — but the propagator does not
know that. Have your assistant rerun the exact propagation with a
*strong* kick ($\kappa = 0.5$, same $\Delta t$ and steps) and compute
the spectrum out to $\omega \approx 2$. Then run the check that is
yours alone: a third-harmonic line — spectral weight near
$3\times$ the fundamental that is absent in the $\kappa = 10^{-3}$
spectrum (compare the two spectra's weight in a window around
$3\omega_1$ with `numpy`, demanding a ratio $> 10$). Odd harmonics
from a centrosymmetric system: nonlinear optics, three lines of
`numpy` away from linear spectroscopy. The check is yours.
:::

## Notebook summary

The volume's exact laboratory learned to move. Its eight lowest states
were classified by exchange and parity — ground state $-2.2391$,
two dipole-active singlet-odd excitations at $0.5339$ and $0.7147$,
a triplet and an even singlet on the forbidden list — and the exact
real-time dynamics honored all of it: Crank–Nicolson with a
once-factorized sparse LU conserved the norm to $10^{-8}$ across
$4000$ steps, the kick response ratio under kick halving was
$2.0000$, linear to well within the $1\%$ tolerance, the spectral
peaks landed on the static excitation energies
at $2\times10^{-4}$, and the forbidden windows stayed fivefold below even the weaker
allowed line — spin and parity acting as selection rules *inside* the
propagator.
Against this certified answer, the two mean-field dynamics took their
grades: time-dependent Hartree–Fock (adiabatic exact exchange for this
system) reproduced the spectrum with a $+3.0\%$ blue shift on a
$-2.2250$ ground state ($14$ mHa of correlation missing — the same
deficit [§8.2](exact-laboratory.ipynb) measured); time-dependent
Hartree, self-interaction left in, landed $62\%$ low with spurious
structure — [§8.8](exact-conditions-band-gap.ipynb)'s static disease
as a dynamical catastrophe. The scoreboard is the point: TDDFT's
accuracy is exactly the accuracy of its functional, and the exact
testing ground is what makes that sentence quantitative.

## Outlook

- Production TDDFT propagates Kohn–Sham orbitals with adiabatic
  LDA/GGA functionals — better than Hartree (self-interaction partly
  cancelled), worse than exact exchange for two electrons; Ullrich
  {cite}`ullrich2012` maps the functional landscape, Casida's
  linear-response matrix reformulates the kick as an eigenproblem, and
  the known failures (double excitations, charge transfer, memory)
  all trace to the adiabatic approximation this notebook priced.
- The kick protocol scales: real codes compute molecular absorption
  exactly as Exercise 3 did, with the 2D grid swapped for Kohn–Sham
  orbitals in 3D.
- One movement remains: from excited electrons to *paired* ones. The
  volume's finale, [§8.17](bcs-superconductivity.ipynb), takes the
  attractive corner of the interaction map and finds the ground state
  nobody predicted: superconductivity.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()